# NLP API processing pipeline

Runs every article in `articles.csv` through AWS Comprehend, Google Cloud NLP, and Azure AI Language and writes the CSVs the bias analysis notebook consumes.

Outputs:
- `results_sentiment.csv` doc-level sentiment, one row per (article, API)
- `results_entities.csv` named entities, with entity-level sentiment where the API exposes it
- `results_keyphrases.csv` key phrases
- `results_azure_opinions.csv` Azure opinion mining (target/assessment pairs)
- `processing_log.csv` per-call status and truncation flags

Credentials need to be set in `.env` before running cell 2.

## Dependencies

In [4]:

pip install boto3 google-cloud-language azure-ai-textanalytics pandas krippendorff statsmodels

Note: you may need to restart the kernel to use updated packages.


## Config and credentials

Paths and keys live in `.env`. Swap in your own before running.

In [5]:
import os
import json
import time
import pandas as pd
from datetime import datetime
from dotenv import load_dotenv

# paths
ARTICLES_PATH = 'articles.csv'
OUTPUT_DIR = 'api_results'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# credentials from .env
load_dotenv()

AWS_ACCESS_KEY = os.environ['AWS_ACCESS_KEY_ID']
AWS_SECRET_KEY = os.environ['AWS_SECRET_ACCESS_KEY']
AWS_REGION     = 'us-east-1'

# google needs the path to the service-account JSON, not the key itself
GOOGLE_CREDENTIALS_PATH = os.environ['GOOGLE_APPLICATION_CREDENTIALS']

AZURE_ENDPOINT = os.environ['AZURE_LANGUAGE_ENDPOINT']
AZURE_KEY      = os.environ['AZURE_LANGUAGE_KEY']

# small sleeps between calls; bump these up if any API starts throttling
DELAY_BETWEEN_ARTICLES = 1.0
DELAY_BETWEEN_APIS     = 0.3

print('Configuration loaded.')
print(f'Articles: {ARTICLES_PATH}')
print(f'Output:   {OUTPUT_DIR}/')

Configuration loaded.
Articles: articles.csv
Output:   api_results/


## Load articles

In [6]:
df = pd.read_csv(ARTICLES_PATH, encoding='utf-8-sig')

# drop rows with no text
df = df[df['text'].notna() & (df['text'].str.strip() != '')].copy()
df = df.reset_index(drop=True)

# outlet names came in with several spellings (BBC vs BBC News, France24 vs France 24,
# strait/Straits/The Straits Times, SCMP vs South China Morning Post). canonicalise here
# so every downstream CSV inherits the same name. bias_analysis, fdi_pipeline and
# mitigation_pipeline import this same dict.
OUTLET_NORMALISATION = {
    'BBC News':           'BBC',
    'France24':           'France 24',
    'Strait Times':       'Straits Times',
    'The Straits Times':  'Straits Times',
    'SCMP':               'South China Morning Post',
}
df['outlet'] = df['outlet'].replace(OUTLET_NORMALISATION)

print(f'Loaded {len(df)} articles with text')
print(f'Topics:  {df["topic"].value_counts().to_dict()}')
print(f'Outlets: {df["outlet"].nunique()} unique outlets (after normalisation)')
print(f'Pairs:   {df["pair_id"].nunique()} event pairs')
print()

# quick peek
df[['article_id', 'pair_id', 'outlet', 'topic', 'frame', 'word_count']].head(10)


Loaded 237 articles with text
Topics:  {'immigration': 88, 'political conflict': 85, 'environment': 64}
Outlets: 19 unique outlets (after normalisation)
Pairs:   20 event pairs



,article_id,pair_id,outlet,topic,frame,word_count
0,art_001,imm_001,Fox News,immigration,security,720
1,art_002,imm_001,BBC,immigration,neutral,723
2,art_003,imm_001,The Guardian,immigration,humanitarian,948
3,art_004,imm_001,CNN,immigration,neutral,1915
4,art_005,imm_001,Reuters,immigration,neutral,807
5,art_006,imm_001,Global Times,immigration,humanitarian,678
6,art_007,imm_001,RT News,immigration,humanitarian,335
7,art_008,imm_001,Al Jazeera,immigration,humanitarian,741
8,art_009,imm_001,Straits Times,immigration,humanitarian,1071
9,art_010,imm_001,Le Monde,immigration,neutral,551


## API clients

Each client is wrapped in try/except so the pipeline still runs if one API is misconfigured. Useful when only one set of credentials is working at a time.

In [7]:
# AWS
aws_client = None
try:
    import boto3
    aws_client = boto3.client(
        'comprehend',
        region_name=AWS_REGION,
        aws_access_key_id=AWS_ACCESS_KEY,
        aws_secret_access_key=AWS_SECRET_KEY
    )
    # ping to confirm creds
    aws_client.detect_sentiment(Text='test', LanguageCode='en')
    print('✓ AWS Comprehend connected')
except Exception as e:
    print(f'✗ AWS Comprehend failed: {e}')
    aws_client = None

# Google
google_client = None
try:
    from google.cloud import language_v1
    os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = GOOGLE_CREDENTIALS_PATH
    google_client = language_v1.LanguageServiceClient()
    # ping
    test_doc = language_v1.Document(content='test', type_=language_v1.Document.Type.PLAIN_TEXT)
    google_client.analyze_sentiment(request={'document': test_doc})
    print('✓ Google Cloud NLP connected')
except Exception as e:
    print(f'✗ Google Cloud NLP failed: {e}')
    google_client = None

# Azure
azure_client = None
try:
    from azure.ai.textanalytics import TextAnalyticsClient
    from azure.core.credentials import AzureKeyCredential
    azure_client = TextAnalyticsClient(
        endpoint=AZURE_ENDPOINT,
        credential=AzureKeyCredential(AZURE_KEY)
    )
    # ping
    azure_client.analyze_sentiment(['test'])
    print('✓ Azure AI Language connected')
except Exception as e:
    print(f'✗ Azure AI Language failed: {e}')
    azure_client = None

active_apis = []
if aws_client:    active_apis.append('AWS')
if google_client: active_apis.append('Google')
if azure_client:  active_apis.append('Azure')
print(f'\nActive APIs: {active_apis}')
if not active_apis:
    print('⚠ No APIs connected! Check your credentials in Cell 2.')

✓ AWS Comprehend connected
✓ Google Cloud NLP connected
✓ Azure AI Language connected

Active APIs: ['AWS', 'Google', 'Azure']


## Processing functions

One function per API, each returns dicts ready for a DataFrame. Per-API input limits: AWS 5,000 bytes, Azure 5,120 chars, Google 1M chars, so we truncate to the AWS limit (and log it). Errors get caught and written to the processing log instead of killing the run.

In [8]:
import re

# cleaning + truncation, applied once so all 3 APIs see the same input

def clean_article_text(text):
    """Strip non-editorial boilerplate. Keep all journalistic content; no stopword or punctuation removal."""
    # contributor bylines at the end
    text = re.sub(
        r'\n.*(?:contributed to this report|contributed reporting)\.?\s*$',
        '', text, flags=re.IGNORECASE
    )
    # trailing pub timestamps
    text = re.sub(
        r'\n\s*(?:Published|Updated|Posted|Modified)\s*[-–:]?\s*\w+ \d{1,2},? \d{4}.*$',
        '', text, flags=re.IGNORECASE
    )
    # newsletter/CTA lines
    text = re.sub(
        r'\n.*(?:Sign up now|Subscribe to|Get the latest|Download our app|Join our).*',
        '', text, flags=re.IGNORECASE
    )
    # 'read more' / 'see also' navigation cruft
    text = re.sub(
        r'\n\s*(?:Read more|See also|ALSO READ|MORE:|Related:).*',
        '', text, flags=re.IGNORECASE | re.MULTILINE
    )
    # collapse runs of blank lines
    text = re.sub(r'\n{3,}', '\n\n', text)
    return text.strip()

def prepare_text(text, max_bytes=4800):
    """Clean boilerplate, then truncate to the AWS 5000-byte limit.

    Returns (prepared_text, was_truncated, original_chars, final_chars).
    The same prepared_text goes to all three APIs so score differences
    come from the APIs and not from different inputs.
    """
    cleaned = clean_article_text(text)
    original_chars = len(cleaned)
    
    encoded = cleaned.encode('utf-8')
    if len(encoded) <= max_bytes:
        return cleaned, False, original_chars, len(cleaned)
    
    # cut at the byte limit, then back off to the last full sentence
    truncated = encoded[:max_bytes].decode('utf-8', errors='ignore')
    last_period = truncated.rfind('.')
    if last_period > len(truncated) // 2:
        truncated = truncated[:last_period + 1]
    
    return truncated, True, original_chars, len(truncated)

def process_aws(text, article_id):
    """Run one article through AWS Comprehend. text must already be passed through prepare_text()."""
    if aws_client is None:
        return None, [], [], {'api': 'AWS', 'article_id': article_id, 'status': 'skipped', 'error': 'client not connected'}

    log = {'api': 'AWS', 'article_id': article_id, 'timestamp': datetime.now().isoformat()}

    try:
        sent_resp = aws_client.detect_sentiment(Text=text, LanguageCode='en')
        sentiment = {
            'article_id': article_id,
            'api': 'AWS',
            'sentiment_label': sent_resp['Sentiment'],
            'score_positive': round(sent_resp['SentimentScore']['Positive'], 4),
            'score_negative': round(sent_resp['SentimentScore']['Negative'], 4),
            'score_neutral':  round(sent_resp['SentimentScore']['Neutral'], 4),
            'score_mixed':    round(sent_resp['SentimentScore']['Mixed'], 4),
            'net_sentiment':  round(sent_resp['SentimentScore']['Positive'] - sent_resp['SentimentScore']['Negative'], 4)
        }
        time.sleep(DELAY_BETWEEN_APIS)

        ent_resp = aws_client.detect_entities(Text=text, LanguageCode='en')
        entities = [
            {
                'article_id': article_id,
                'api': 'AWS',
                'entity_text': e['Text'],
                'entity_type': e['Type'],
                'confidence': round(e['Score'], 4),
                'entity_sentiment_score': None,
                'entity_salience': None
            }
            for e in ent_resp['Entities']
        ]
        time.sleep(DELAY_BETWEEN_APIS)

        kp_resp = aws_client.detect_key_phrases(Text=text, LanguageCode='en')
        keyphrases = [
            {
                'article_id': article_id,
                'api': 'AWS',
                'phrase': p['Text'],
                'confidence': round(p['Score'], 4)
            }
            for p in kp_resp['KeyPhrases']
        ]

        log['status'] = 'success'
        log['entities_found'] = len(entities)
        log['keyphrases_found'] = len(keyphrases)
        return sentiment, entities, keyphrases, log

    except Exception as e:
        log['status'] = 'error'
        log['error'] = str(e)
        return None, [], [], log

def process_google(text, article_id):
    """Run one article through Google Cloud NLP. Native entity sentiment here feeds IAEI."""
    if google_client is None:
        return None, [], [], {'api': 'Google', 'article_id': article_id, 'status': 'skipped', 'error': 'client not connected'}

    log = {'api': 'Google', 'article_id': article_id, 'timestamp': datetime.now().isoformat()}

    try:
        document = language_v1.Document(content=text, type_=language_v1.Document.Type.PLAIN_TEXT)

        sent_resp = google_client.analyze_sentiment(request={'document': document})
        doc_sent = sent_resp.document_sentiment
        sentiment = {
            'article_id': article_id,
            'api': 'Google',
            'sentiment_label': (
                'VERY_NEGATIVE' if doc_sent.score < -0.5 else
                'NEGATIVE' if doc_sent.score < -0.1 else
                'NEUTRAL' if doc_sent.score < 0.1 else
                'POSITIVE' if doc_sent.score < 0.5 else 'VERY_POSITIVE'
            ),
            'score_positive': max(doc_sent.score, 0),
            'score_negative': abs(min(doc_sent.score, 0)),
            'score_neutral':  None,
            'score_mixed':    None,
            'net_sentiment':  round(doc_sent.score, 4),
            'google_magnitude': round(doc_sent.magnitude, 4)
        }
        time.sleep(DELAY_BETWEEN_APIS)

        ent_resp = google_client.analyze_entity_sentiment(request={'document': document})
        entities = [
            {
                'article_id': article_id,
                'api': 'Google',
                'entity_text': e.name,
                'entity_type': language_v1.Entity.Type(e.type_).name,
                'confidence': None,
                'entity_sentiment_score': round(e.sentiment.score, 4),
                'entity_sentiment_magnitude': round(e.sentiment.magnitude, 4),
                'entity_salience': round(e.salience, 4)
            }
            for e in ent_resp.entities
        ]

        # Google has no key-phrase API, so reuse top-salience entities as a proxy
        keyphrases = [
            {
                'article_id': article_id,
                'api': 'Google',
                'phrase': e.name,
                'confidence': round(e.salience, 4)
            }
            for e in sorted(ent_resp.entities, key=lambda x: x.salience, reverse=True)[:15]
        ]

        log['status'] = 'success'
        log['entities_found'] = len(entities)
        log['keyphrases_found'] = len(keyphrases)
        return sentiment, entities, keyphrases, log

    except Exception as e:
        log['status'] = 'error'
        log['error'] = str(e)
        return None, [], [], log

def process_azure(text, article_id):
    """Run one article through Azure. Opinion mining gives target/assessment pairs that feed the framing analysis."""
    if azure_client is None:
        return None, [], [], {'api': 'Azure', 'article_id': article_id, 'status': 'skipped', 'error': 'client not connected'}, []

    log = {'api': 'Azure', 'article_id': article_id, 'timestamp': datetime.now().isoformat()}

    try:
        # sentiment + opinion mining in one call
        sent_result = azure_client.analyze_sentiment([text], show_opinion_mining=True)
        doc = sent_result[0]
        if doc.is_error:
            raise Exception(f'Azure returned error: {doc.error}')

        sentiment = {
            'article_id': article_id,
            'api': 'Azure',
            'sentiment_label': doc.sentiment.upper(),
            'score_positive': round(doc.confidence_scores.positive, 4),
            'score_negative': round(doc.confidence_scores.negative, 4),
            'score_neutral':  round(doc.confidence_scores.neutral, 4),
            'score_mixed':    None,
            'net_sentiment':  round(doc.confidence_scores.positive - doc.confidence_scores.negative, 4)
        }

        # opinion Mining
        opinions = []
        for sent_idx, sentence in enumerate(doc.sentences):
            for opinion in (sentence.mined_opinions or []):
                for assessment in opinion.assessments:
                    opinions.append({
                        'article_id': article_id,
                        'sentence_idx': sent_idx,
                        'target_text': opinion.target.text,
                        'target_sentiment': opinion.target.sentiment,
                        'assessment_text': assessment.text,
                        'assessment_sentiment': assessment.sentiment,
                        'sentence_sentiment': sentence.sentiment,
                        'sentence_text': sentence.text[:200]
                    })
        time.sleep(DELAY_BETWEEN_APIS)

        ner_result = azure_client.recognize_entities([text])
        ner_doc = ner_result[0]
        entities = []
        if not ner_doc.is_error:
            entities = [
                {
                    'article_id': article_id,
                    'api': 'Azure',
                    'entity_text': e.text,
                    'entity_type': e.category,
                    'entity_subtype': e.subcategory,
                    'confidence': round(e.confidence_score, 4),
                    'entity_sentiment_score': None,
                    'entity_salience': None
                }
                for e in ner_doc.entities
            ]
        time.sleep(DELAY_BETWEEN_APIS)

        kp_result = azure_client.extract_key_phrases([text])
        kp_doc = kp_result[0]
        keyphrases = []
        if not kp_doc.is_error:
            keyphrases = [
                {
                    'article_id': article_id,
                    'api': 'Azure',
                    'phrase': p,
                    'confidence': None
                }
                for p in kp_doc.key_phrases
            ]

        log['status'] = 'success'
        log['entities_found'] = len(entities)
        log['keyphrases_found'] = len(keyphrases)
        log['opinions_found'] = len(opinions)
        return sentiment, entities, keyphrases, log, opinions

    except Exception as e:
        log['status'] = 'error'
        log['error'] = str(e)
        return None, [], [], log, []

print('Processing functions loaded.')
print('Text cleaning: contributor lines, timestamps, CTAs, related links')
print('All 3 APIs receive identical text per article (truncated to AWS limit if needed)')

Processing functions loaded.
Text cleaning: contributor lines, timestamps, CTAs, related links
All 3 APIs receive identical text per article (truncated to AWS limit if needed)


## Main loop

Iterates over every article and hits whichever APIs are connected.

In [9]:
# bump this if the run dies partway and you need to resume from a specific index
START_FROM = 0

all_sentiments = []
all_entities   = []
all_keyphrases = []
all_opinions   = []   # Azure only
all_logs       = []

total = len(df)
start_time = time.time()

print(f'Processing {total} articles through {len(active_apis)} APIs...')
print(f'Started at: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}\n')

for idx in range(START_FROM, total):
    row = df.iloc[idx]
    article_id = row['article_id']
    raw_text   = str(row['text'])

    # clean + truncate once; same text goes to every API
    prepared_text, was_truncated, orig_chars, final_chars = prepare_text(raw_text)

    trunc_flag = f' [TRUNCATED {orig_chars}→{final_chars} chars]' if was_truncated else ''
    print(f'[{idx+1}/{total}] {article_id} | {row["outlet"]:20s} | {row["topic"]:20s} | {row["frame"]}{trunc_flag}')

    # attach truncation info to every API's log entry
    trunc_meta = {
        'truncated': was_truncated,
        'original_chars': orig_chars,
        'chars_sent': final_chars
    }

    if aws_client:
        sent, ents, kps, log = process_aws(prepared_text, article_id)
        log.update(trunc_meta)
        if sent: all_sentiments.append(sent)
        all_entities.extend(ents)
        all_keyphrases.extend(kps)
        all_logs.append(log)
        status = '✓' if log['status'] == 'success' else '✗'
        print(f'         AWS:    {status} | {log.get("entities_found", 0)} entities, {log.get("keyphrases_found", 0)} phrases')

    if google_client:
        sent, ents, kps, log = process_google(prepared_text, article_id)
        log.update(trunc_meta)
        if sent: all_sentiments.append(sent)
        all_entities.extend(ents)
        all_keyphrases.extend(kps)
        all_logs.append(log)
        status = '✓' if log['status'] == 'success' else '✗'
        print(f'         Google: {status} | {log.get("entities_found", 0)} entities')

    if azure_client:
        sent, ents, kps, log, opinions = process_azure(prepared_text, article_id)
        log.update(trunc_meta)
        if sent: all_sentiments.append(sent)
        all_entities.extend(ents)
        all_keyphrases.extend(kps)
        all_opinions.extend(opinions)
        all_logs.append(log)
        status = '✓' if log['status'] == 'success' else '✗'
        print(f'         Azure:  {status} | {log.get("entities_found", 0)} entities, {log.get("opinions_found", 0)} opinions')

    time.sleep(DELAY_BETWEEN_ARTICLES)

elapsed = time.time() - start_time
print(f'\n{"="*60}')
print(f'DONE in {elapsed:.1f} seconds')
print(f'Sentiments: {len(all_sentiments)} | Entities: {len(all_entities)} | Key Phrases: {len(all_keyphrases)} | Opinions: {len(all_opinions)}')

# how many articles got truncated overall
trunc_count = sum(1 for l in all_logs if l.get('truncated') and l['api'] == active_apis[0])
print(f'\nArticles truncated: {trunc_count}/{total}')
if trunc_count > 0:
    print('(Same truncated text was sent to ALL APIs for fair comparison)')

Processing 237 articles through 3 APIs...
Started at: 2026-05-23 00:16:08

[1/237] art_001 | Fox News             | immigration          | security
         AWS:    ✓ | 77 entities, 177 phrases
         Google: ✓ | 166 entities
         Azure:  ✓ | 135 entities, 1 opinions
[2/237] art_002 | BBC                  | immigration          | neutral
         AWS:    ✓ | 78 entities, 183 phrases
         Google: ✓ | 164 entities
         Azure:  ✓ | 119 entities, 2 opinions
[3/237] art_003 | The Guardian         | immigration          | humanitarian [TRUNCATED 5827→4702 chars]
         AWS:    ✓ | 104 entities, 208 phrases
         Google: ✓ | 170 entities
         Azure:  ✓ | 156 entities, 3 opinions
[4/237] art_004 | CNN                  | immigration          | neutral [TRUNCATED 11589→4717 chars]
         AWS:    ✓ | 66 entities, 185 phrases
         Google: ✓ | 178 entities
         Azure:  ✓ | 130 entities, 1 opinions
[5/237] art_005 | Reuters              | immigration          | neutr

## Save results

Dump everything to CSVs. Each one is merged with the article metadata (outlet, topic, frame, pair_id) so downstream filtering doesn't need an extra join.

In [10]:
# metadata to merge into every results file
meta_cols = ['article_id', 'pair_id', 'outlet', 'topic', 'frame']
meta = df[meta_cols].copy()

In [11]:

def save_with_metadata(data_list, filename, merge_on='article_id'):
    """List of dicts -> DataFrame, merge with article metadata, save to OUTPUT_DIR."""
    if not data_list:
        print(f'  ⚠ {filename}: No data to save')
        return None
    result_df = pd.DataFrame(data_list)
    result_df = result_df.merge(meta, on=merge_on, how='left')
    path = os.path.join(OUTPUT_DIR, filename)
    result_df.to_csv(path, index=False, encoding='utf-8-sig')
    print(f'  ✓ {filename}: {len(result_df)} rows saved')
    return result_df

print('Saving results...\n')

df_sentiments  = save_with_metadata(all_sentiments, 'results_sentiment.csv')
df_entities    = save_with_metadata(all_entities, 'results_entities.csv')
df_keyphrases  = save_with_metadata(all_keyphrases, 'results_keyphrases.csv')
df_opinions    = save_with_metadata(all_opinions, 'results_azure_opinions.csv')

# log doesn't need the meta merge
df_logs = pd.DataFrame(all_logs)
df_logs.to_csv(os.path.join(OUTPUT_DIR, 'processing_log.csv'), index=False)
print(f'  ✓ processing_log.csv: {len(df_logs)} rows saved')

print(f'\nAll files saved to: {OUTPUT_DIR}/')

Saving results...

  ✓ results_sentiment.csv: 711 rows saved
  ✓ results_entities.csv: 76133 rows saved
  ✓ results_keyphrases.csv: 65226 rows saved
  ✓ results_azure_opinions.csv: 444 rows saved
  ✓ processing_log.csv: 711 rows saved

All files saved to: api_results/


## Quick sanity check

Eyeballs the saved results so I know if anything is obviously off before moving on.

In [12]:
if df_sentiments is not None:
    print('SENTIMENT RESULTS SUMMARY')
    print('=' * 50)
    print(f'Total rows: {len(df_sentiments)}')
    print(f'\nRows per API:')
    print(df_sentiments['api'].value_counts().to_string())
    print(f'\nSentiment labels by API:')
    print(df_sentiments.groupby('api')['sentiment_label'].value_counts().to_string())
    print(f'\nNet sentiment stats by API:')
    print(df_sentiments.groupby('api')['net_sentiment'].describe().round(3).to_string())
    print()

# any errors logged?
if df_logs is not None:
    errors = df_logs[df_logs['status'] == 'error']
    if len(errors) > 0:
        print(f'\n⚠ {len(errors)} ERRORS occurred:')
        print(errors[['api', 'article_id', 'error']].to_string())
    else:
        print('✓ No errors during processing')

    # truncation summary
    truncated = df_logs[df_logs.get('truncated', pd.Series([False])) == True]
    if len(truncated) > 0:
        print(f'\n⚠ {len(truncated)} articles were truncated (too long for API limit):')
        print(truncated[['api', 'article_id', 'chars_sent']].to_string())

SENTIMENT RESULTS SUMMARY
Total rows: 711

Rows per API:
api
AWS       237
Google    237
Azure     237

Sentiment labels by API:
api     sentiment_label
AWS     NEUTRAL            228
        MIXED                6
        NEGATIVE             3
Azure   NEGATIVE           168
        MIXED               66
        POSITIVE             2
        NEUTRAL              1
Google  NEGATIVE           212
        NEUTRAL             13
        VERY_NEGATIVE       11
        POSITIVE             1

Net sentiment stats by API:
        count   mean    std    min    25%   50%    75%    max
api                                                          
AWS     237.0 -0.058  0.115 -0.659 -0.098 -0.03  0.001  0.266
Azure   237.0 -0.727  0.215 -0.980 -0.840 -0.79 -0.690  0.820
Google  237.0 -0.307  0.147 -0.700 -0.400 -0.30 -0.200  0.100

✓ No errors during processing

⚠ 297 articles were truncated (too long for API limit):
        api article_id  chars_sent
6       AWS    art_003        4702
7    Goog

## Metrics preview

IPSV (inter-platform sentiment variance), IAEI (intra-article entity inconsistency, Google native + AWS/Azure projected), plus cross-platform agreement broken out by frame and topic.

## Normalising sentiment scores

AWS and Azure define `net_sentiment` as `positive - negative` (a difference of confidence probs), but Google returns a polarity score on [-1, +1]. The ranges look similar but they're not the same quantity.

z-scoring each API's scores across the corpus is what makes IPSV and Krippendorff's alpha valid here, otherwise the metrics would partly reflect scale differences instead of real disagreement.

In [13]:
# z-score per API so IPSV and Krippendorff's alpha capture disagreement
# instead of the AWS/Azure-vs-Google scale difference. raw scores kept around too.

if df_sentiments is not None and len(df_sentiments) > 0:

    df_sentiments['net_sentiment_z'] = df_sentiments.groupby('api')['net_sentiment'].transform(
        lambda x: (x - x.mean()) / x.std() if x.std() > 0 else 0.0
    )

    print('Sentiment Score Normalisation')
    print('=' * 50)
    print('\nRaw net_sentiment stats per API:')
    print(df_sentiments.groupby('api')['net_sentiment'].describe().round(4).to_string())
    print('\nZ-normalised net_sentiment_z stats per API:')
    print(df_sentiments.groupby('api')['net_sentiment_z'].describe().round(4).to_string())
    print('\n(z-scores have mean≈0, std≈1 within each API — now comparable)')
else:
    print('No sentiment data to normalise.')


Sentiment Score Normalisation

Raw net_sentiment stats per API:
        count    mean     std     min     25%     50%     75%     max
api                                                                  
AWS     237.0 -0.0583  0.1148 -0.6588 -0.0976 -0.0299  0.0009  0.2661
Azure   237.0 -0.7272  0.2148 -0.9800 -0.8400 -0.7900 -0.6900  0.8200
Google  237.0 -0.3072  0.1473 -0.7000 -0.4000 -0.3000 -0.2000  0.1000

Z-normalised net_sentiment_z stats per API:
        count  mean  std     min     25%     50%     75%     max
api                                                             
AWS     237.0   0.0  1.0 -5.2309 -0.3421  0.2476  0.5159  2.8261
Azure   237.0   0.0  1.0 -1.1769 -0.5251 -0.2923  0.1733  7.2031
Google  237.0   0.0  1.0 -2.6676 -0.6304  0.0487  0.7278  2.7650

(z-scores have mean≈0, std≈1 within each API — now comparable)


In [14]:
import numpy as np

if df_sentiments is not None and len(df_sentiments) > 0:

    # IPSV = per-article sample std of the per-API sentiment score (Eq 4.1).
    # used to be .var(); switched to .std() to match the methodology.
    # z-scored scores so the std reflects disagreement and not the AWS/Azure-vs-Google scale gap.

    score_col = 'net_sentiment_z' if 'net_sentiment_z' in df_sentiments.columns else 'net_sentiment'
    score_label = 'z-normalised' if score_col == 'net_sentiment_z' else 'raw'

    pivot = df_sentiments.pivot_table(
        index='article_id',
        columns='api',
        values=score_col
    )
    pivot['IPSV'] = pivot[active_apis].std(axis=1)

    ipsv_df = pivot.reset_index().merge(meta, on='article_id')

    print(f'IPSV (Inter-Platform Sentiment Variance) — using {score_label} scores')
    print('=' * 60)
    print('Higher IPSV = more disagreement between APIs\n')
    print(ipsv_df.groupby('topic')['IPSV'].describe().round(4).to_string())
    print()

    print('\nIPSV by Frame')
    print('=' * 50)
    print(ipsv_df.groupby('frame')['IPSV'].describe().round(4).to_string())
    print()

    # which articles do the APIs disagree on most?
    print('\nTop 5 Most Disagreed-Upon Articles (highest IPSV):')
    top_disagree = ipsv_df.nlargest(5, 'IPSV')[['article_id', 'outlet', 'topic', 'frame', 'IPSV'] + active_apis]
    print(top_disagree.to_string(index=False))

    ipsv_path = os.path.join(OUTPUT_DIR, 'results_ipsv.csv')
    ipsv_df.to_csv(ipsv_path, index=False, encoding='utf-8-sig')
    print(f'\n\u2713 results_ipsv.csv saved')

else:
    print('No sentiment data available yet.')


IPSV (Inter-Platform Sentiment Variance) — using z-normalised scores
Higher IPSV = more disagreement between APIs

                    count    mean     std     min     25%     50%     75%     max
topic                                                                            
environment          64.0  0.9167  0.6080  0.0784  0.5264  0.7786  1.1224  3.0972
immigration          88.0  0.7263  0.5280  0.0867  0.3737  0.6291  0.9180  3.4969
political conflict   85.0  0.5169  0.2816  0.0613  0.3348  0.4637  0.6608  1.5414


IPSV by Frame
                count    mean     std     min     25%     50%     75%     max
frame                                                                        
concern          41.0  0.8070  0.4273  0.0784  0.4447  0.7615  1.0673  1.8782
humanitarian     31.0  0.6192  0.3009  0.1312  0.4110  0.5944  0.7808  1.4693
neutral          95.0  0.7530  0.6311  0.0635  0.3676  0.6144  0.9232  3.4969
people-centric   28.0  0.5904  0.2720  0.0613  0.3807  0.5539  0.7660

In [15]:
# IAEI = intra-article entity inconsistency.
# for article i on platform p: mean over entities of |s_{e,p} - S_{i,p}|.
# Google has native entity sentiment so we use that directly.
# AWS/Azure don't, so we project: for each entity, find the sentences
# containing it, score each sentence via the API, and average. A sentence
# that mentions several entities contributes the same score to each
# (standard attribution). Restricted to PERSON/ORGANIZATION for the
# projected case since those are what framing actually targets.

import re
import time
from collections import defaultdict

iaei_frames = []

# cache sentence-level scores so duplicates only cost one API call
SENTENCE_SENT_CACHE = {'AWS': {}, 'Azure': {}}
TARGET_ENTITY_TYPES_PROJECTED = {'PERSON', 'ORGANIZATION', 'Person', 'Organization'}

def _split_sentences(text):
    """Cheap regex sentence splitter; same one used elsewhere."""
    return [s.strip() for s in re.split(r'(?<=[.!?])\s+', text) if s.strip()]

def _score_sentence_aws(sentence):
    """AWS net_sentiment (positive - negative) for a single sentence, cached."""
    if sentence in SENTENCE_SENT_CACHE['AWS']:
        return SENTENCE_SENT_CACHE['AWS'][sentence]
    if aws_client is None:
        return None
    try:
        # 5000-byte limit is way above any sentence we'll see
        resp = aws_client.detect_sentiment(Text=sentence, LanguageCode='en')
        s = resp['SentimentScore']
        net = s['Positive'] - s['Negative']
    except Exception as e:
        net = None
    SENTENCE_SENT_CACHE['AWS'][sentence] = net
    time.sleep(DELAY_BETWEEN_APIS)
    return net

def _score_sentence_azure(sentence):
    """Azure net_sentiment (positive - negative) for a single sentence, cached."""
    if sentence in SENTENCE_SENT_CACHE['Azure']:
        return SENTENCE_SENT_CACHE['Azure'][sentence]
    if azure_client is None:
        return None
    try:
        resp = azure_client.analyze_sentiment([sentence])[0]
        if getattr(resp, 'is_error', False):
            net = None
        else:
            cs = resp.confidence_scores
            net = cs.positive - cs.negative
    except Exception as e:
        net = None
    SENTENCE_SENT_CACHE['Azure'][sentence] = net
    time.sleep(DELAY_BETWEEN_APIS)
    return net

SCORE_SENTENCE = {'AWS': _score_sentence_aws, 'Azure': _score_sentence_azure}

# Google: native entity sentiment
if df_entities is not None and 'Google' in active_apis:
    google_ents = df_entities[
        (df_entities['api'] == 'Google') &
        (df_entities['entity_sentiment_score'].notna())
    ].copy()

    google_doc_sent = df_sentiments[df_sentiments['api'] == 'Google'][['article_id', 'net_sentiment']]
    google_doc_sent = google_doc_sent.rename(columns={'net_sentiment': 'doc_sentiment'})

    google_ents = google_ents.merge(google_doc_sent, on='article_id', how='left')
    google_ents['entity_doc_diff'] = abs(google_ents['entity_sentiment_score'] - google_ents['doc_sentiment'])

    iaei_google = google_ents.groupby('article_id')['entity_doc_diff'].mean().reset_index()
    iaei_google.columns = ['article_id', 'IAEI']
    iaei_google['api'] = 'Google'
    iaei_google['method'] = 'native_entity_sentiment'
    iaei_frames.append(iaei_google)

# AWS / Azure: sentence-context projection (described above)
for api_name in [a for a in active_apis if a in ('AWS', 'Azure')]:
    if df_entities is None or df_sentiments is None:
        continue

    score_fn = SCORE_SENTENCE[api_name]
    api_ents = df_entities[
        (df_entities['api'] == api_name) &
        (df_entities['entity_type'].isin(TARGET_ENTITY_TYPES_PROJECTED))
    ].copy()
    if len(api_ents) == 0:
        continue

    api_doc_sent = df_sentiments[df_sentiments['api'] == api_name][['article_id', 'net_sentiment']]
    api_doc_sent = api_doc_sent.rename(columns={'net_sentiment': 'doc_sentiment'})

    print(f'\nProjecting sentence-level sentiment for {api_name} '
          f'({len(api_ents)} target entities across '
          f'{api_ents["article_id"].nunique()} articles)...')

    article_ids = sorted(api_ents['article_id'].unique())
    entity_diffs = []

    for art_idx, article_id in enumerate(article_ids):
        art_ents = api_ents[api_ents['article_id'] == article_id]
        doc_row = api_doc_sent[api_doc_sent['article_id'] == article_id]
        if len(doc_row) == 0:
            continue
        doc_sent_val = float(doc_row['doc_sentiment'].values[0])

        text_rows = df[df['article_id'] == article_id]
        if len(text_rows) == 0:
            continue
        article_text = clean_article_text(str(text_rows.iloc[0]['text']))
        # use the prepared text so the sentence pool matches what the API actually saw
        prepared_text, _, _, _ = prepare_text(str(text_rows.iloc[0]['text']))
        sentences = _split_sentences(prepared_text)

        # entity -> matching sentences (case-insensitive substring).
        # cache means each unique sentence costs exactly one API call regardless
        # of how many entities it gets attributed to.
        for _, ent_row in art_ents.iterrows():
            ent_text = str(ent_row['entity_text']).strip()
            if not ent_text:
                continue
            ent_lower = ent_text.lower()
            matching = [s for s in sentences if ent_lower in s.lower()]
            if not matching:
                continue
            sent_scores = [score_fn(s) for s in matching]
            sent_scores = [v for v in sent_scores if v is not None]
            if not sent_scores:
                continue
            attributed_sent = sum(sent_scores) / len(sent_scores)
            entity_diffs.append({
                'article_id':       article_id,
                'entity_text':      ent_text,
                'entity_type':      ent_row['entity_type'],
                'attributed_sent':  attributed_sent,
                'doc_sentiment':    doc_sent_val,
                'entity_doc_diff':  abs(attributed_sent - doc_sent_val),
                'n_sentences':      len(matching),
            })

        if (art_idx + 1) % 25 == 0:
            print(f'  {api_name}: processed {art_idx + 1}/{len(article_ids)} articles '
                  f'(cache size: {len(SENTENCE_SENT_CACHE[api_name])} unique sentences)')

    if not entity_diffs:
        print(f'  {api_name}: no entity-level scores produced.')
        continue

    ent_diff_df = pd.DataFrame(entity_diffs)
    iaei_proj = ent_diff_df.groupby('article_id')['entity_doc_diff'].mean().reset_index()
    iaei_proj.columns = ['article_id', 'IAEI']
    iaei_proj['api'] = api_name
    iaei_proj['method'] = 'sentence_context_projection'
    iaei_frames.append(iaei_proj)

    # also dump the per-entity intermediate so I can sanity-check later
    ent_diff_df.to_csv(
        os.path.join(OUTPUT_DIR, f'results_iaei_{api_name.lower()}_per_entity.csv'),
        index=False, encoding='utf-8-sig'
    )
    print(f'  {api_name}: IAEI computed for {len(iaei_proj)} articles '
          f'(mean = {iaei_proj["IAEI"].mean():.4f}, '
          f'{len(SENTENCE_SENT_CACHE[api_name])} unique sentences scored)')

# ── Combine and display ──────────────────────────────────────────
if iaei_frames:
    iaei_all = pd.concat(iaei_frames, ignore_index=True)
    iaei_all = iaei_all.merge(meta, on='article_id')

    print('\n' + '=' * 60)
    print('IAEI (Intra-Article Entity Inconsistency) — All APIs')
    print('=' * 60)
    print('Higher IAEI = neutral overall tone but targeted entity framing')
    print('Google = native entity sentiment | AWS/Azure = sentence-context projection\n')
    print(iaei_all.groupby(['api', 'topic'])['IAEI'].describe().round(4).to_string())
    print()
    print('\nIAEI by API and Frame')
    print(iaei_all.groupby(['api', 'frame'])['IAEI'].describe().round(4).to_string())

    # sanity check: AWS/Azure IAEI should NOT be perfectly correlated with
    # |document sentiment| anymore. (The old proxy was r = 1.000.)
    print('\nSanity check — IAEI vs |document sentiment| correlation per API:')
    print('(Pre-fix this was r = 1.000 for AWS and Azure; should now be < 1.)')
    for api in iaei_all['api'].unique():
        sub = iaei_all[iaei_all['api'] == api][['article_id', 'IAEI']].drop_duplicates()
        doc = df_sentiments[df_sentiments['api'] == api][['article_id', 'net_sentiment']]
        m = sub.merge(doc, on='article_id')
        if len(m) > 2:
            r = m['IAEI'].corr(m['net_sentiment'].abs())
            print(f'  {api}: r = {r:.4f}  (n = {len(m)})')

    iaei_path = os.path.join(OUTPUT_DIR, 'results_iaei.csv')
    iaei_all.to_csv(iaei_path, index=False, encoding='utf-8-sig')
    print(f'\n✓ results_iaei.csv: {len(iaei_all)} rows saved')
else:
    print('IAEI requires entity data from at least one API.')



Projecting sentence-level sentiment for AWS (7645 target entities across 236 articles)...
  AWS: processed 25/236 articles (cache size: 518 unique sentences)
  AWS: processed 50/236 articles (cache size: 982 unique sentences)
  AWS: processed 75/236 articles (cache size: 1348 unique sentences)
  AWS: processed 100/236 articles (cache size: 1715 unique sentences)
  AWS: processed 125/236 articles (cache size: 2113 unique sentences)
  AWS: processed 150/236 articles (cache size: 2546 unique sentences)
  AWS: processed 175/236 articles (cache size: 3017 unique sentences)
  AWS: processed 200/236 articles (cache size: 3422 unique sentences)
  AWS: processed 225/236 articles (cache size: 3746 unique sentences)
  AWS: IAEI computed for 236 articles (mean = 0.1006, 3853 unique sentences scored)

Projecting sentence-level sentiment for Azure (4854 target entities across 236 articles)...
  Azure: processed 25/236 articles (cache size: 399 unique sentences)
  Azure: processed 50/236 articles (c

## Lexical Frame Indicator (LFI)

A score based on the presence of words from pre-compiled lexicons associated with specific frames (e.g., humanitarian vs. security frames for immigration).

These are starter lists that can be refined after reviewing key phrases output.

In [16]:
# FRAME LEXICONS - Expand these based on corpus review
# each topic has two contrasting frame lexicons.

import re
import spacy
from scipy.stats import pearsonr

# one-time: pip install spacy && python -m spacy download en_core_web_sm
try:
    nlp = spacy.load('en_core_web_sm', disable=['parser', 'ner'])
except OSError:
    raise RuntimeError(
        "spaCy model missing. Run: python -m spacy download en_core_web_sm"
    )

FRAME_LEXICONS = {
    'immigration': {
        'humanitarian': [
            'refugee', 'asylum seeker', 'asylum', 'displaced', 'flee',
            'persecution', 'human rights', 'humanitarian', 'families',
            'children', 'vulnerable', 'protection', 'shelter', 'aid',
            'crisis', 'desperate', 'seeking safety', 'undocumented',
            'migrant workers', 'dream', 'opportunity'
        ],
        'security': [
            'illegal', 'illegals', 'alien', 'invasion', 'border security',
            'enforcement', 'deportation', 'deport', 'criminal', 'gang',
            'smuggling', 'trafficking', 'threat', 'surge', 'flood',
            'overwhelm', 'overrun', 'national security', 'crackdown',
            'raid', 'detain', 'arrest', 'trespass', 'lawbreaker'
        ]
    },
    'environment': {
        'concern': [
            'crisis', 'emergency', 'catastrophe', 'disaster', 'extinction',
            'irreversible', 'tipping point', 'devastating', 'urgent',
            'scientists warn', 'unprecedented', 'accelerating', 'threat',
            'carbon emissions', 'fossil fuels', 'greenhouse', 'pollution',
            'warming', 'rising sea levels', 'extreme weather', 'biodiversity',
            'ecosystem', 'sustainability', 'renewable', 'clean energy'
        ],
        'skeptical': [
            'alarmist', 'hoax', 'natural cycle', 'exaggerated', 'costly',
            'regulation', 'economic impact', 'job losses', 'burden',
            'overreach', 'uncertain', 'debate', 'models', 'prediction',
            'agenda', 'ideology', 'taxpayer'
        ]
    },
    'political conflict': {
        'people-centric': [
            'protesters', 'demonstrators', 'activists', 'peaceful',
            'movement', 'grassroots', 'civil rights', 'freedom fighters',
            'liberation', 'resistance', 'solidarity', 'uprising',
            'civilians', 'victims', 'displaced', 'suffering',
            'human rights', 'oppression', 'persecution', 'atrocities',
            'massacre', 'brutality', 'injustice', 'freedom',
            'self-determination', 'occupied', 'besieged'
        ],
        'state-centric': [
            'terrorists', 'insurgents', 'militants', 'extremists',
            'rioters', 'agitators', 'separatists', 'radicals',
            'aggression', 'provocation', 'destabilise', 'subversion',
            'national security', 'sovereignty', 'territorial integrity',
            'law and order', 'counterterrorism', 'crackdown',
            'retaliation', 'deterrence', 'regime change', 'threat',
            'hostile', 'illegal', 'incitement', 'propaganda'
        ]
    }
}


In [17]:
def lemmatize_string(text):
    """Return text as a space-joined string of lowercased lemmas."""
    doc = nlp(text.lower())
    return ' '.join(tok.lemma_ for tok in doc if not tok.is_space)

# pre-lemmatize all lexicon terms once. Deduplicate so plurals
# collapsing to singular (e.g., 'illegal' + 'illegals' → 'illegal')
# don't cause double-counting.
LEMMA_LEXICONS = {
    topic: {
        frame: sorted(set(lemmatize_string(t) for t in terms))
        for frame, terms in frames.items()
    }
    for topic, frames in FRAME_LEXICONS.items()
}

# quick visibility: what did lemmatization collapse?
print("Lexicon size before/after lemmatization & dedup:")
for topic in FRAME_LEXICONS:
    for frame in FRAME_LEXICONS[topic]:
        n_before = len(FRAME_LEXICONS[topic][frame])
        n_after = len(LEMMA_LEXICONS[topic][frame])
        if n_before != n_after:
            print(f"  {topic}/{frame}: {n_before} → {n_after}")

# ─── Batch-lemmatize all article texts (uses nlp.pipe for speed) ───
print("\nLemmatizing 237 articles...")
texts = [str(t) for t in df['text'].tolist()]
lemma_texts = []
for doc in nlp.pipe([t.lower() for t in texts], batch_size=20):
    lemma_texts.append(' '.join(tok.lemma_ for tok in doc if not tok.is_space))
df = df.copy()
df['_text_lemma'] = lemma_texts
print("Done.")

# ─── LFI computation (both methods) ────────────────────────────────
def _count(text, terms):
    return sum(len(re.findall(r'\b' + re.escape(t) + r'\b', text))
               for t in terms)

def compute_lfi(text_raw, text_lemma, topic):
    if topic not in FRAME_LEXICONS:
        return {k: None for k in [
            'lfi_score_exact', 'lfi_score_lemma',
            'frame_a_count_exact', 'frame_b_count_exact',
            'frame_a_count_lemma', 'frame_b_count_lemma',
            'lfi_total_hits_exact', 'lfi_total_hits_lemma',
            'frame_a_label', 'frame_b_label']}

    a, b = list(FRAME_LEXICONS[topic].keys())

    # exact match
    ax_e = _count(text_raw.lower(), FRAME_LEXICONS[topic][a])
    bx_e = _count(text_raw.lower(), FRAME_LEXICONS[topic][b])
    tot_e = ax_e + bx_e
    s_e = round((ax_e - bx_e) / tot_e, 4) if tot_e > 0 else 0.0

    # lemma match
    ax_l = _count(text_lemma, LEMMA_LEXICONS[topic][a])
    bx_l = _count(text_lemma, LEMMA_LEXICONS[topic][b])
    tot_l = ax_l + bx_l
    s_l = round((ax_l - bx_l) / tot_l, 4) if tot_l > 0 else 0.0

    return {
        'frame_a_label': a, 'frame_b_label': b,
        'lfi_score_exact': s_e, 'lfi_score_lemma': s_l,
        'frame_a_count_exact': ax_e, 'frame_b_count_exact': bx_e,
        'frame_a_count_lemma': ax_l, 'frame_b_count_lemma': bx_l,
        'lfi_total_hits_exact': tot_e, 'lfi_total_hits_lemma': tot_l,
    }

lfi_rows = []
for _, row in df.iterrows():
    result = compute_lfi(str(row['text']), row['_text_lemma'], row['topic'])
    result['article_id'] = row['article_id']
    lfi_rows.append(result)

df_lfi = pd.DataFrame(lfi_rows).merge(meta, on='article_id')

# primary score column for downstream analyses (uses lemma version)
df_lfi['lfi_score'] = df_lfi['lfi_score_lemma']
df_lfi['frame_a_count'] = df_lfi['frame_a_count_lemma']
df_lfi['frame_b_count'] = df_lfi['frame_b_count_lemma']
df_lfi['lfi_total_hits'] = df_lfi['lfi_total_hits_lemma']

# save
lfi_path = os.path.join(OUTPUT_DIR, 'results_lfi.csv')
df_lfi.to_csv(lfi_path, index=False, encoding='utf-8-sig')
print(f'\n✓ results_lfi.csv: {len(df_lfi)} rows saved')

# ─── Method comparison: exact vs lemma ─────────────────────────────
print('\nLFI METHOD COMPARISON: Exact vs Lemma')
print('=' * 60)
print(f"Mean total hits per article:")
print(f"  Exact:   {df_lfi['lfi_total_hits_exact'].mean():.2f}")
print(f"  Lemma:   {df_lfi['lfi_total_hits_lemma'].mean():.2f}")
uplift = (df_lfi['lfi_total_hits_lemma'].mean()
          / max(df_lfi['lfi_total_hits_exact'].mean(), 1e-9) - 1) * 100
print(f"  Lemma uplift: {uplift:+.1f}%")

mask = df_lfi['lfi_score_exact'].notna() & df_lfi['lfi_score_lemma'].notna()
r, p = pearsonr(df_lfi.loc[mask, 'lfi_score_exact'],
                df_lfi.loc[mask, 'lfi_score_lemma'])
print(f"\nCorrelation between exact and lemma LFI: r = {r:.4f} (p = {p:.4f})")
print("(High r → lemmatization preserves rank ordering;")
print(" lower r → lemmatization meaningfully changes the metric.)")

print('\nLFI by topic and frame (lemma-based, primary):')
for topic in df_lfi['topic'].unique():
    subset = df_lfi[df_lfi['topic'] == topic]
    if subset['lfi_score_lemma'].isna().all():
        continue
    a = subset['frame_a_label'].iloc[0]
    b = subset['frame_b_label'].iloc[0]
    print(f"\n{topic} (+ = {a}, − = {b}):")
    print(subset.groupby('frame')['lfi_score_lemma'].describe().round(4).to_string())

Lexicon size before/after lemmatization & dedup:
  immigration/security: 24 → 23

Lemmatizing 237 articles...
Done.

✓ results_lfi.csv: 237 rows saved

LFI METHOD COMPARISON: Exact vs Lemma
Mean total hits per article:
  Exact:   6.35
  Lemma:   8.70
  Lemma uplift: +36.9%

Correlation between exact and lemma LFI: r = 0.8532 (p = 0.0000)
(High r → lemmatization preserves rank ordering;
 lower r → lemmatization meaningfully changes the metric.)

LFI by topic and frame (lemma-based, primary):

immigration (+ = humanitarian, − = security):
              count    mean     std  min     25%     50%     75%  max
frame                                                                
humanitarian   31.0  0.3395  0.6575 -1.0  0.0000  0.6000  0.8575  1.0
neutral        38.0  0.1353  0.5792 -1.0 -0.0441  0.2500  0.4286  1.0
security       19.0  0.3225  0.5405 -0.9  0.0000  0.3846  0.7461  1.0

political conflict (+ = people-centric, − = state-centric):
                count    mean     std     min 

## Cross-Platform Reliability (Krippendorff's Alpha)

Agreement across the three systems will be measured using Krippendorff's alpha, with each API treated as an annotator.
- α ≥ 0.80 → strong reliability
- α ≥ 0.67 → acceptable for tentative conclusions
- α < 0.67 → low reliability, suggests systematic bias across the ecosystem

In [18]:
# pip install krippendorff (uncomment and run once if needed)

In [19]:

try:
    import krippendorff
    _has_krippendorff = True
except ImportError:
    print('\u26a0 Install krippendorff: pip install krippendorff')
    _has_krippendorff = False

if _has_krippendorff and df_sentiments is not None and len(active_apis) >= 2:

    # FIX: Use z-normalised scores for valid cross-platform comparison
    score_col = 'net_sentiment_z' if 'net_sentiment_z' in df_sentiments.columns else 'net_sentiment'
    score_label = 'z-normalised' if score_col == 'net_sentiment_z' else 'raw'

    # build the reliability matrix: rows = annotators (APIs), columns = articles
    pivot_alpha = df_sentiments.pivot_table(
        index='api', columns='article_id', values=score_col
    )
    reliability_data = pivot_alpha.values  # shape: (n_apis, n_articles)

    # overall alpha
    alpha_overall = krippendorff.alpha(
        reliability_data=reliability_data,
        level_of_measurement='interval'
    )
    print(f'CROSS-PLATFORM RELIABILITY (Krippendorff\'s Alpha) — {score_label} scores')
    print('=' * 65)
    print(f'Overall \u03b1 = {alpha_overall:.4f}', end='  ')
    if alpha_overall >= 0.80:
        print('\u2192 Strong reliability')
    elif alpha_overall >= 0.67:
        print('\u2192 Acceptable for tentative conclusions')
    else:
        print('\u2192 Low reliability \u2014 suggests systematic bias')
    print()

    # alpha by topic
    print('Alpha by Topic:')
    print('-' * 40)
    for topic in df['topic'].unique():
        topic_articles = meta[meta['topic'] == topic]['article_id'].values
        topic_cols = [c for c in pivot_alpha.columns if c in topic_articles]
        if len(topic_cols) < 2:
            continue
        topic_data = pivot_alpha[topic_cols].values
        alpha_topic = krippendorff.alpha(
            reliability_data=topic_data,
            level_of_measurement='interval'
        )
        print(f'  {topic:25s} \u03b1 = {alpha_topic:.4f}')

    # alpha by frame
    print(f'\nAlpha by Frame:')
    print('-' * 40)
    for frame in df['frame'].unique():
        frame_articles = meta[meta['frame'] == frame]['article_id'].values
        frame_cols = [c for c in pivot_alpha.columns if c in frame_articles]
        if len(frame_cols) < 2:
            continue
        frame_data = pivot_alpha[frame_cols].values
        alpha_frame = krippendorff.alpha(
            reliability_data=frame_data,
            level_of_measurement='interval'
        )
        print(f'  {frame:25s} \u03b1 = {alpha_frame:.4f}')

elif len(active_apis) < 2:
    print('Krippendorff\'s alpha requires at least 2 active APIs.')
else:
    print('No sentiment data available.')


CROSS-PLATFORM RELIABILITY (Krippendorff's Alpha) — z-normalised scores
Overall α = 0.2505  → Low reliability — suggests systematic bias

Alpha by Topic:
----------------------------------------
  immigration               α = 0.2665
  political conflict        α = 0.2677
  environment               α = 0.1818

Alpha by Frame:
----------------------------------------
  security                  α = 0.3140
  neutral                   α = 0.2875
  humanitarian              α = 0.0761
  people-centric            α = 0.1852
  state-centric             α = 0.1354
  concern                   α = 0.1307
  skeptical                 α = 0.2231


## Statistical Analysis (Proposal Section 4.3.5)

1. **Paired t-tests** - Do contrasting frames produce significantly different sentiment scores?
2. **Cohen's d** - How large is the effect?
3. **Pearson correlation** - Association between linguistic features (LFI) and API sentiment scores

In [20]:
from scipy import stats

# FRAMING-EFFECT TESTS (methodology-aligned contrasts)
# for each topic we test the principal frame contrast that the
# methodology specifies, EXCLUDING neutral as a baseline class:
# • immigration: humanitarian vs security
# • environment: concern vs skeptical
# • political conflict: people-centric vs state-centric
# test: Welch's two-sample t-test (equal_var=False) because frame
# is between-article and variances are heterogeneous by design.
# effect: Cohen's d with pooled SD; small=0.2 / medium=0.5 / large=0.8.
# multiple testing: Benjamini-Hochberg FDR at alpha=0.05 across the
# full grid of 3 topics × 3 APIs = 9 tests.
# why not paired: the events in this corpus do not contain matched
# numbers of articles across the two principal frames, so pairing
# by pair_id discards most data; Welch on the full per-frame samples
# is the correct test for the methodology's contrast.

def cohens_d(group1, group2):
    n1, n2 = len(group1), len(group2)
    var1, var2 = group1.var(ddof=1), group2.var(ddof=1)
    pooled = np.sqrt(((n1 - 1) * var1 + (n2 - 1) * var2) / (n1 + n2 - 2))
    return 0.0 if pooled == 0 else (group1.mean() - group2.mean()) / pooled

# methodology's principal frame contrasts, in fixed order so the sign
# of Cohen's d has a consistent interpretation across topics.
FRAME_CONTRASTS = {
    'immigration':        ('humanitarian',   'security'),
    'environment':        ('concern',        'skeptical'),
    'political conflict': ('people-centric', 'state-centric'),
}

if df_sentiments is not None:

    print('FRAMING-EFFECT TESTS (Welch\'s t, methodology contrasts)')
    print('=' * 65)

    stat_rows = []

    for api_name in active_apis:
        api_sent = df_sentiments[df_sentiments['api'] == api_name][
            ['article_id', 'net_sentiment']
        ].merge(meta, on='article_id')

        print(f'\n--- {api_name} ---')

        for topic, (frame_a, frame_b) in FRAME_CONTRASTS.items():
            topic_data = api_sent[api_sent['topic'] == topic]

            group_a = topic_data[topic_data['frame'] == frame_a]['net_sentiment'].dropna()
            group_b = topic_data[topic_data['frame'] == frame_b]['net_sentiment'].dropna()

            if len(group_a) < 2 or len(group_b) < 2:
                print(f'  {topic} ({frame_a} vs {frame_b}): '
                      f'insufficient data (n_a={len(group_a)}, n_b={len(group_b)})')
                continue

            # welch's t-test
            t_stat, p_value = stats.ttest_ind(group_a, group_b, equal_var=False)
            d = cohens_d(group_a, group_b)

            sig = '***' if p_value < 0.001 else '**' if p_value < 0.01 else '*' if p_value < 0.05 else 'ns'
            effect = ('large'      if abs(d) >= 0.8
                      else 'medium' if abs(d) >= 0.5
                      else 'small'  if abs(d) >= 0.2
                      else 'negligible')

            print(f'  {topic} ({frame_a} vs {frame_b}):')
            print(f'    Mean A={group_a.mean():+.4f} (n={len(group_a)}), '
                  f'Mean B={group_b.mean():+.4f} (n={len(group_b)})')
            print(f'    t={t_stat:+.4f}, p={p_value:.4f} {sig}, '
                  f'd={d:+.4f} ({effect})')

            stat_rows.append({
                'api':         api_name,
                'topic':       topic,
                'frame_a':     frame_a,
                'frame_b':     frame_b,
                'mean_a':      round(group_a.mean(), 4),
                'mean_b':      round(group_b.mean(), 4),
                'n_a':         len(group_a),
                'n_b':         len(group_b),
                'test_type':   'welch_t',
                't_statistic': round(t_stat, 4),
                'p_value_raw': round(p_value, 6),
                'cohens_d':    round(d, 4),
                'effect_size': effect,
            })

    # ── Benjamini-Hochberg FDR correction across the full 3×3 grid ──
    if stat_rows:
        stat_df = pd.DataFrame(stat_rows).sort_values('p_value_raw').reset_index(drop=True)
        m = len(stat_df)
        # BH-FDR adjusted p-values: p_adj_i = min_{j>=i}( m * p_(j) / j ), monotone
        bh = (stat_df['p_value_raw'] * m / (stat_df.index + 1)).clip(upper=1.0)
        # enforce monotonicity from largest rank down
        bh_adj = bh.iloc[::-1].cummin().iloc[::-1]
        stat_df['p_value_corrected']    = bh_adj.round(6)
        stat_df['significant_after_fdr'] = stat_df['p_value_corrected'] < 0.05
        stat_df['significance'] = stat_df['p_value_corrected'].apply(
            lambda p: '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'ns'
        )
        stat_df = stat_df.sort_values(['api', 'topic']).reset_index(drop=True)

        print('\n' + '=' * 65)
        print('After Benjamini–Hochberg FDR correction (alpha = 0.05):')
        print('=' * 65)
        print(stat_df[['api', 'topic', 'frame_a', 'frame_b', 't_statistic',
                       'p_value_raw', 'p_value_corrected',
                       'significant_after_fdr', 'cohens_d', 'effect_size']]
              .to_string(index=False))

        out_path = os.path.join(OUTPUT_DIR, 'results_statistical_tests.csv')
        stat_df.to_csv(out_path, index=False, encoding='utf-8-sig')
        print(f'\n✓ results_statistical_tests.csv saved ({len(stat_df)} tests)')


FRAMING-EFFECT TESTS (Welch's t, methodology contrasts)

--- AWS ---
  immigration (humanitarian vs security):
    Mean A=-0.0268 (n=31), Mean B=-0.0415 (n=19)
    t=+0.5942, p=0.5559 ns, d=+0.1733 (negligible)
  environment (concern vs skeptical):
    Mean A=-0.0713 (n=41), Mean B=-0.3426 (n=3)
    t=+1.8588, p=0.1998 ns, d=+2.0586 (large)
  political conflict (people-centric vs state-centric):
    Mean A=-0.1037 (n=28), Mean B=-0.0417 (n=20)
    t=-2.7488, p=0.0085 **, d=-0.7698 (medium)

--- Google ---
  immigration (humanitarian vs security):
    Mean A=-0.3419 (n=31), Mean B=-0.3158 (n=19)
    t=-0.7368, p=0.4661 ns, d=-0.2195 (small)
  environment (concern vs skeptical):
    Mean A=-0.2049 (n=41), Mean B=-0.3333 (n=3)
    t=+1.4246, p=0.2803 ns, d=+1.0524 (large)
  political conflict (people-centric vs state-centric):
    Mean A=-0.3857 (n=28), Mean B=-0.3000 (n=20)
    t=-2.6144, p=0.0120 *, d=-0.7187 (medium)

--- Azure ---
  immigration (humanitarian vs security):
    Mean A=-

In [21]:
# PEARSON CORRELATION: LFI vs API Sentiment (proposal sec. 4.3.5)
# tests whether linguistic framing features (measured by LFI)
# are associated with sentiment scores from each API.

if df_sentiments is not None and 'df_lfi' in dir() and df_lfi is not None:

    print('PEARSON CORRELATION: LFI Score vs API Sentiment')
    print('=' * 55)
    print('Positive r = humanitarian/urgency/protest framing \u2192 higher sentiment\n')

    corr_rows = []

    for api_name in active_apis:
        api_sent = df_sentiments[df_sentiments['api'] == api_name][['article_id', 'net_sentiment']]
        merged = api_sent.merge(
            df_lfi[['article_id', 'lfi_score', 'topic']],
            on='article_id'
        ).dropna(subset=['lfi_score'])

        if len(merged) < 3:
            continue

        # overall correlation
        r, p = stats.pearsonr(merged['lfi_score'], merged['net_sentiment'])
        print(f'{api_name} (overall): r={r:.4f}, p={p:.4f}')
        corr_rows.append({'api': api_name, 'topic': 'overall', 'pearson_r': round(r, 4), 'p_value': round(p, 4), 'n': len(merged)})

        # per topic
        for topic in merged['topic'].unique():
            subset = merged[merged['topic'] == topic]
            if len(subset) < 3:
                continue
            r_t, p_t = stats.pearsonr(subset['lfi_score'], subset['net_sentiment'])
            print(f'  \u2514\u2500 {topic}: r={r_t:.4f}, p={p_t:.4f} (n={len(subset)})')
            corr_rows.append({'api': api_name, 'topic': topic, 'pearson_r': round(r_t, 4), 'p_value': round(p_t, 4), 'n': len(subset)})

    if corr_rows:
        df_corr = pd.DataFrame(corr_rows)
        corr_path = os.path.join(OUTPUT_DIR, 'results_correlations.csv')
        df_corr.to_csv(corr_path, index=False)
        print(f'\n\u2713 results_correlations.csv: {len(df_corr)} rows saved')
else:
    print('Need both sentiment data and LFI data to compute correlations.')

PEARSON CORRELATION: LFI Score vs API Sentiment
Positive r = humanitarian/urgency/protest framing → higher sentiment

AWS (overall): r=-0.1194, p=0.0666
  └─ immigration: r=-0.1205, p=0.2636 (n=88)
  └─ political conflict: r=0.1156, p=0.2923 (n=85)
  └─ environment: r=-0.2019, p=0.1097 (n=64)
Google (overall): r=0.0460, p=0.4814
  └─ immigration: r=-0.2787, p=0.0086 (n=88)
  └─ political conflict: r=0.0948, p=0.3882 (n=85)
  └─ environment: r=-0.0084, p=0.9474 (n=64)
Azure (overall): r=-0.1210, p=0.0630
  └─ immigration: r=-0.0999, p=0.3545 (n=88)
  └─ political conflict: r=0.2291, p=0.0350 (n=85)
  └─ environment: r=-0.5910, p=0.0000 (n=64)

✓ results_correlations.csv: 12 rows saved


## Truncation Impact Analysis

Articles exceeding the 4,800-byte limit are truncated to the first ~700-800 words. Since Chen et al. (2020) showed that framing cues concentrate in opening paragraphs, truncation may systematically bias results. This cell tests whether truncated articles show different metric distributions.

In [22]:
# TRUNCATION IMPACT ANALYSIS
# the methodology truncates to AWS's 5KB limit. Since framing
# cues concentrate early in articles (Chen et al., 2020), this is a
# potential confound. This analysis checks whether truncation status
# correlates with the metrics.

if df_sentiments is not None:
    # get truncation status from logs
    trunc_log = pd.DataFrame(all_logs)
    trunc_status = trunc_log[trunc_log['api'] == active_apis[0]][['article_id', 'truncated']].drop_duplicates()

    # merge with IPSV
    if 'ipsv_df' in dir() and ipsv_df is not None:
        ipsv_trunc = ipsv_df.merge(trunc_status, on='article_id', how='left')
        print('TRUNCATION IMPACT ON IPSV')
        print('=' * 50)
        print(ipsv_trunc.groupby('truncated')['IPSV'].describe().round(4).to_string())

        # test if truncated articles have significantly different IPSV
        trunc_group = ipsv_trunc[ipsv_trunc['truncated'] == True]['IPSV'].dropna()
        notrunc_group = ipsv_trunc[ipsv_trunc['truncated'] == False]['IPSV'].dropna()
        if len(trunc_group) >= 2 and len(notrunc_group) >= 2:
            t, p = stats.ttest_ind(trunc_group, notrunc_group)
            print(f'\nt-test (truncated vs not): t={t:.4f}, p={p:.4f}')
            if p < 0.05:
                print('⚠ Truncation significantly affects IPSV — report this as a limitation')
            else:
                print('✓ No significant truncation effect on IPSV')
        print()

    # merge with LFI
    if 'df_lfi' in dir() and df_lfi is not None:
        lfi_trunc = df_lfi.merge(trunc_status, on='article_id', how='left')
        print('\nTRUNCATION IMPACT ON LFI')
        print('=' * 50)
        print(lfi_trunc.groupby('truncated')['lfi_score'].describe().round(4).to_string())

    # summary
    print(f'\nTruncation summary: {len(trunc_status[trunc_status["truncated"]==True])}/{len(trunc_status)} articles truncated')
    print('Note: Truncated text may capture more opening-paragraph framing cues')
    print('(Chen et al., 2020). Consider this when interpreting results.')
else:
    print('No data available for truncation analysis.')


TRUNCATION IMPACT ON IPSV
           count    mean     std     min     25%     50%     75%     max
truncated                                                               
False      138.0  0.7350  0.4946  0.0613  0.3853  0.6225  0.9235  3.0007
True        99.0  0.6575  0.5176  0.0635  0.3467  0.5566  0.7965  3.4969

t-test (truncated vs not): t=-1.1669, p=0.2444
✓ No significant truncation effect on IPSV


TRUNCATION IMPACT ON LFI
           count    mean     std  min     25%     50%     75%  max
truncated                                                         
False      138.0  0.2744  0.6439 -1.0 -0.0441  0.3333  1.0000  1.0
True        99.0  0.3205  0.6241 -1.0  0.0000  0.4000  0.8229  1.0

Truncation summary: 99/237 articles truncated
Note: Truncated text may capture more opening-paragraph framing cues
(Chen et al., 2020). Consider this when interpreting results.


## Geographic & Cultural Bias Analysis (Proposal Section 4.3.4)

Groups API outputs by source region to identify whether commercial NLP services show systematic biases linked to the geographic or cultural origin of content.

In [24]:
# GEOGRAPHIC & CULTURAL BIAS ANALYSIS (Proposal Section 4.3.4)
# OUTLET_REGIONS keys must match the *canonical* outlet names applied
# in cell 6 (OUTLET_NORMALISATION). See also the OUTLET_COUNTRY /
# COUNTRY_REGION mapping in bias_analysis.ipynb cell 4 - both use the
# same canonical names.

OUTLET_REGIONS = {
    'Reuters':                  'Global',
    'BBC':                      'Western (UK)',
    'The Guardian':             'Western (UK)',
    'CNN':                      'Western (US)',
    'Fox News':                 'Western (US)',
    'Le Monde':                 'Western (Europe)',
    'France 24':                'Western (Europe)',
    'DW':                       'Western (Europe)',
    'Al Jazeera':               'Non-Western (Middle East)',
    'RT News':                  'Non-Western (Russia)',
    'Global Times':             'Non-Western (China)',
    'Xinhua':                   'Non-Western (China)',
    'China Daily':              'Non-Western (China)',
    'South China Morning Post': 'Non-Western (East Asia)',
    'Straits Times':            'Non-Western (Southeast Asia)',
    'The Hindu':                'Non-Western (South Asia)',
    'Indian Express':           'Non-Western (South Asia)',
    'News24':                   'Non-Western (Africa)',
    'Daily Maverick':           'Non-Western (Africa)',
}

# broader grouping for statistical power
OUTLET_BROAD_REGION = {}
for outlet, region in OUTLET_REGIONS.items():
    if region.startswith('Western') or region == 'Global':
        OUTLET_BROAD_REGION[outlet] = 'Western/Global'
    else:
        OUTLET_BROAD_REGION[outlet] = 'Non-Western'

if df_sentiments is not None:
    # df_sentiments already carries outlet/topic/frame (added by
    # save_with_metadata in cell 15). DO NOT re-merge meta - that
    # would create outlet_x/outlet_y. Just use the existing columns.
    sent_geo = df_sentiments.copy()

    if 'outlet' not in sent_geo.columns:
        # defensive fallback in case df_sentiments was rebuilt without metadata.
        sent_geo = sent_geo.merge(meta, on='article_id', how='left')

    sent_geo['region']       = sent_geo['outlet'].map(OUTLET_REGIONS).fillna('Unknown')
    sent_geo['broad_region'] = sent_geo['outlet'].map(OUTLET_BROAD_REGION).fillna('Unknown')

    # sanity check: any outlets that fell through the region map are a sign
    # of an unhandled spelling - report them up front instead of silently
    # bucketing them as Unknown.
    unmapped = sorted(sent_geo.loc[sent_geo['region'] == 'Unknown', 'outlet'].unique())
    if unmapped:
        print(f'⚠ Outlets with no region mapping: {unmapped}')
        print('   Add them to OUTLET_REGIONS above to include in the analysis.\n')

    print('GEOGRAPHIC & CULTURAL BIAS ANALYSIS')
    print('=' * 60)

    # 1. Regional Performance Comparison
    print('\n1. Sentiment by Region and API')
    print('-' * 50)
    print(sent_geo.groupby(['api', 'broad_region'])['net_sentiment'].describe().round(4).to_string())

    # 2. Test for significant regional differences per API (Welch's t)
    print('\n\n2. Western vs Non-Western Sentiment Difference (per API)')
    print('-' * 55)
    for api_name in active_apis:
        api_data = sent_geo[sent_geo['api'] == api_name]
        western = api_data[api_data['broad_region'] == 'Western/Global']['net_sentiment'].dropna()
        non_western = api_data[api_data['broad_region'] == 'Non-Western']['net_sentiment'].dropna()
        if len(western) >= 2 and len(non_western) >= 2:
            t, p = stats.ttest_ind(western, non_western, equal_var=False)
            d = cohens_d(western, non_western)
            sig = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'ns'
            print(f'  {api_name}: Western mean={western.mean():.4f}, Non-Western mean={non_western.mean():.4f}')
            print(f'    t={t:.4f}, p={p:.4f} {sig}, Cohen\'s d={d:.4f}')

    # 3. IPSV by region - do APIs disagree more on non-Western content?
    if 'ipsv_df' in dir() and ipsv_df is not None:
        ipsv_geo = ipsv_df.copy()
        if 'outlet' not in ipsv_geo.columns:
            ipsv_geo = ipsv_geo.merge(meta, on='article_id', how='left')
        ipsv_geo['broad_region'] = ipsv_geo['outlet'].map(OUTLET_BROAD_REGION).fillna('Unknown')
        print('\n\n3. IPSV by Region (do APIs disagree more on non-Western content?)')
        print('-' * 60)
        print(ipsv_geo.groupby('broad_region')['IPSV'].describe().round(4).to_string())

    # 4. Detailed breakdown by specific region
    print('\n\n4. Sentiment by Specific Region (all APIs combined)')
    print('-' * 50)
    region_summary = sent_geo.groupby('region')['net_sentiment'].agg(['mean', 'std', 'count']).round(4)
    print(region_summary.sort_values('mean').to_string())

    # save
    geo_cols = ['article_id', 'api', 'net_sentiment', 'outlet', 'topic', 'frame', 'region', 'broad_region']
    geo_path = os.path.join(OUTPUT_DIR, 'results_geographic_bias.csv')
    sent_geo[[c for c in geo_cols if c in sent_geo.columns]].to_csv(geo_path, index=False, encoding='utf-8-sig')
    print(f'\n✓ results_geographic_bias.csv saved')

else:
    print('No sentiment data available for geographic analysis.')


GEOGRAPHIC & CULTURAL BIAS ANALYSIS

1. Sentiment by Region and API
--------------------------------------------------
                       count    mean     std     min     25%     50%     75%     max
api    broad_region                                                                 
AWS    Non-Western     116.0 -0.0541  0.1213 -0.6588 -0.0840 -0.0278  0.0010  0.2661
       Western/Global  121.0 -0.0624  0.1086 -0.4020 -0.1256 -0.0331  0.0005  0.2061
Azure  Non-Western     116.0 -0.7155  0.2590 -0.9800 -0.8500 -0.7800 -0.6900  0.8200
       Western/Global  121.0 -0.7384  0.1619 -0.9100 -0.8400 -0.8000 -0.6800 -0.0500
Google Non-Western     116.0 -0.2966  0.1577 -0.7000 -0.4000 -0.3000 -0.2000  0.1000
       Western/Global  121.0 -0.3174  0.1364 -0.7000 -0.4000 -0.3000 -0.2000  0.0000


2. Western vs Non-Western Sentiment Difference (per API)
-------------------------------------------------------
  AWS: Western mean=-0.0624, Non-Western mean=-0.0541
    t=-0.5503, p=0.5826 ns, Cohe

## What's Next?

With the output CSVs, you can now:

1. **Paired article analysis** - Filter `results_sentiment.csv` by `pair_id` to compare how different framings of the same event are scored
2. **IPSV analysis** - `results_ipsv.csv` shows where APIs disagree most (z-normalised for valid comparison)
3. **IAEI analysis** - `results_iaei.csv` reveals targeted entity framing (Google native + AWS/Azure proxy)
4. **LFI analysis** - `results_lfi.csv` quantifies framing lexicon presence (word-boundary matched)
5. **Geographic bias** - `results_geographic_bias.csv` compares Western vs Non-Western treatment
6. **Statistical tests** - `results_statistical_tests.csv` includes paired t-tests with FDR correction
7. **Correlations** - `results_correlations.csv` links LFI to API sentiment

**Mitigation pipeline** (separate notebook): Take the highest-IPSV articles and run them through the post-processing neutralisation pipeline (Proposal Section 4.4).

In [25]:
"""
IAEI Summary Statistics for Section 4.9
=========================================
Intra-Article Entity Inconsistency
section of the Results chapter.

"""

import pandas as pd
import numpy as np
from scipy import stats

# 1. LOAD DATA
# adjust the path if you're running outside the project folder
iaei = pd.read_csv("api_results/results_iaei.csv")
ipsv = pd.read_csv("api_results/results_ipsv.csv")

# map outlets to Western / Non-Western (mirrors your region analysis elsewhere)
WESTERN = {
    "BBC", "CNN", "Fox News", "The Guardian", "Reuters", "Le Monde",
    "France 24", "France24", "DW", "News24", "Daily Maverick",
}
iaei["region"] = iaei["outlet"].apply(
    lambda o: "Western" if o in WESTERN else "Non-Western"
)

print("=" * 70)
print("IAEI SUMMARY STATISTICS — FOR SECTION 4.9")
print("=" * 70)
print(f"\nTotal IAEI records: {len(iaei)}")
print(f"Unique articles: {iaei['article_id'].nunique()}")
print(f"APIs covered: {sorted(iaei['api'].unique())}")
print(f"Methods: {sorted(iaei['method'].unique())}")

# 2. OVERALL IAEI PER API (headline numbers for §4.9 ¶1)
print("\n\n[§4.9 ¶1] IAEI descriptive statistics by API")
print("-" * 70)
per_api = iaei.groupby("api")["IAEI"].describe().round(4)
print(per_api.to_string())

# 3. IAEI PER TOPIC PER API (to test whether specific domains drive IAEI)
print("\n\n[§4.9 ¶2] IAEI by topic × API (mean ± std)")
print("-" * 70)
by_topic = iaei.groupby(["api", "topic"])["IAEI"].agg(
    ["count", "mean", "std", "median"]
).round(4)
print(by_topic.to_string())

# 4. IAEI PER FRAME PER API (tests whether framed articles have higher
# entity-document divergence, which would support the "neutral tone
# masking targeted negativity" hypothesis)
print("\n\n[§4.9 ¶2] IAEI by frame × API (mean)")
print("-" * 70)
by_frame = iaei.groupby(["api", "frame"])["IAEI"].agg(
    ["count", "mean", "std"]
).round(4)
print(by_frame.to_string())

# 5. IAEI BY REGION (does cultural origin move IAEI even if it doesn't
# move document sentiment?)
print("\n\n[§4.9 ¶3] IAEI by region × API (Western vs Non-Western)")
print("-" * 70)
by_region = iaei.groupby(["api", "region"])["IAEI"].agg(
    ["count", "mean", "std"]
).round(4)
print(by_region.to_string())

print("\nWelch's t-test: Western vs Non-Western IAEI, per API")
print("-" * 70)
for api in sorted(iaei["api"].unique()):
    sub = iaei[iaei["api"] == api]
    w = sub[sub["region"] == "Western"]["IAEI"].dropna()
    n = sub[sub["region"] == "Non-Western"]["IAEI"].dropna()
    if len(w) > 1 and len(n) > 1:
        t, p = stats.ttest_ind(w, n, equal_var=False)
        pooled_std = np.sqrt((w.var() + n.var()) / 2)
        d = (w.mean() - n.mean()) / pooled_std if pooled_std else np.nan
        print(f"  {api:8s}  t = {t:+.3f}  p = {p:.4f}  "
              f"d = {d:+.3f}  (W n={len(w)}, NW n={len(n)})")

# 6. RELATIONSHIP BETWEEN IAEI AND IPSV
# does the "within-article disagreement" signal align with the
# "across-API disagreement" signal? If yes, IAEI captures a coherent
# bias dimension rather than noise.
print("\n\n[§4.9 ¶4] Correlation between per-article IAEI and IPSV")
print("-" * 70)
# average IAEI per article across APIs (one value per article)
iaei_per_article = iaei.groupby("article_id")["IAEI"].mean().reset_index()
iaei_per_article.columns = ["article_id", "IAEI_mean"]

merged = ipsv[["article_id", "IPSV"]].merge(iaei_per_article, on="article_id")
r, p = stats.pearsonr(merged["IAEI_mean"], merged["IPSV"])
print(f"  Pearson r = {r:+.4f}  p = {p:.4f}  n = {len(merged)}")

# also per API - does each platform's IAEI track its own IPSV contribution?
print("\nPer-API Pearson r (IAEI vs IPSV):")
for api in sorted(iaei["api"].unique()):
    api_iaei = iaei[iaei["api"] == api][["article_id", "IAEI"]]
    m = ipsv[["article_id", "IPSV"]].merge(api_iaei, on="article_id")
    if len(m) > 3:
        r, p = stats.pearsonr(m["IAEI"], m["IPSV"])
        print(f"  {api:8s}  r = {r:+.4f}  p = {p:.4f}  n = {len(m)}")

# 7. TOP 5 HIGHEST-IAEI ARTICLES PER API (qualitative illustration)
print("\n\n[§4.9 optional] Top 5 articles with highest IAEI per API")
print("-" * 70)
for api in sorted(iaei["api"].unique()):
    print(f"\n  {api}:")
    top = iaei[iaei["api"] == api].nlargest(5, "IAEI")[
        ["article_id", "outlet", "topic", "frame", "IAEI"]
    ]
    print(top.to_string(index=False))

# 8. ONE-WAY ANOVA: Does IAEI differ by API? (expected: yes, because of
# the native/proxy operationalisation difference)
print("\n\n[§4.9 caveat] One-way ANOVA on IAEI across APIs")
print("-" * 70)
groups = [iaei[iaei["api"] == a]["IAEI"].dropna() for a in sorted(iaei["api"].unique())]
f_stat, p_val = stats.f_oneway(*groups)
print(f"  F = {f_stat:.3f}  p = {p_val:.4g}")
print("  Interpretation: a significant F here is EXPECTED and does not")
print("  indicate bias — it reflects the native-vs-proxy operationalisation")
print("  difference. Interpret IAEI comparisons *within* each method only.")

print("\n" + "=" * 70)
print("DONE — plug these numbers into Section 4.9 of your draft.")
print("=" * 70)

IAEI SUMMARY STATISTICS — FOR SECTION 4.9

Total IAEI records: 679
Unique articles: 237
APIs covered: ['AWS', 'Azure', 'Google']
Methods: ['native_entity_sentiment', 'sentence_context_projection']


[§4.9 ¶1] IAEI descriptive statistics by API
----------------------------------------------------------------------
        count    mean     std     min     25%     50%     75%     max
api                                                                  
AWS     236.0  0.1006  0.0775  0.0073  0.0526  0.0792  0.1222  0.5710
Azure   206.0  0.3902  0.1812  0.0214  0.2580  0.3801  0.4888  1.3933
Google  237.0  0.2931  0.1306  0.0000  0.2016  0.2862  0.3743  0.7000


[§4.9 ¶2] IAEI by topic × API (mean ± std)
----------------------------------------------------------------------
                           count    mean     std  median
api    topic                                            
AWS    environment            63  0.1427  0.1149  0.1062
       immigration            88  0.0874  0.0529